# 20 · Travel time band, checked against an independent threshold

## What this checks

**Travel time band** keeps the street junctions that can be reached within a stated number of minutes. It is given
junctions already walked from a site, each with the distance travelled to it. This check covers release 0.3.0.

Choosing the junctions is a simple threshold, so that arithmetic is not the interesting part. The interesting part
is a **design claim**. The widget does not ask again for the walking speed or the direction. It reads both from the
junctions it is given. So a band cannot disagree with the walk that produced it.

## How it is checked

That claim can be tested. We work out the band's distance budget again, from the speed carried **on the junctions**.
We take nothing from any setting of our own. Then we select the junctions on that budget. This needs no outside
library: a band is a threshold on distances the junctions already carry. No code is shared with Fieldwork.

The data are 706 junctions of Soho's streets from the John Snow example. They were walked for 1,125 m from one pump,
and the band is 5 minutes.

## What should happen

1. **The band budget** is the minutes × the speed on the junctions, within 5 millimetres.
2. **The same junctions** are in the band, exactly.
3. **Each junction's remaining budget** is worked out against the band, within 5 millimetres. It must not be carried
   over from the longer walk.
4. **The counts** of junctions offered and selected match exactly.
5. **Every junction kept** is marked with the band.
6. **The band is nested, not a ring.** No junction is left out for being too close.

The tolerances are half of the last digit the widget reports, which is two decimal places of a metre. These
criteria were fixed before the check was run.

In [ ]:
# No geometry library is needed: a band is a threshold on distances the junctions already carry.
print("Nothing to load.")

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_20_band import run

report = run(Path.cwd() / "fixtures" / "band-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"])
print(f"  band                  {report['bandMinutes']} min")
print(f"  speed, from the junctions   {report['speedFromTheJunctions']} m/min")
print(f"  direction, from the junctions {report['directionFromTheJunctions']}")
print(f"  band budget, reference      {report['referenceBandBudgetMetres']:.2f} m")
print(f"  band budget, fieldwork      {report['fieldworkBandBudgetMetres']:.2f} m")
print(f"  traversal budget            {report['traversalBudgetMetres']:.2f} m")
print()
print(f"  junctions offered     {report['junctionsOffered']}")
print(f"  junctions in band     {report['junctionsInBand']}")
print(f"  worst remaining-budget error  {report['worstRemainingBudgetErrorMetres']:.3e} m")
print(f"  nested, not a ring    {report['nestedNotARing']}")

## The result on record

First recorded on 11 October 2026, against release 0.3.0. Rerun on 26 November 2026, against release 0.3.0:
**Agrees**.

Both sides kept 134 of the 706 junctions. That is fewer than the 142 first recorded, because
**Reachable junctions** now spends the walk from the pump onto the street network before any street is walked. The band budget was 375.00 m on both sides: 5 minutes at the 75 m a minute
carried on the junctions. The largest difference in any remaining budget was about 5.7 × 10⁻¹⁴ m. The band was nested,
not a ring, and every kept junction was marked with it.

Running the cell above repeats the check on the same data.

## The part most likely to be got wrong

A junction's **remaining budget is worked out again against the band**. It is not carried over from the walk. A
junction 300 m along a 375 m band has 75 m left of the band. It does not have the 825 m it had left of the 1,125 m
walk it came from.

That matters because **Reachable corridor** uses the remaining budget to decide how far into each street a person
gets. Carrying over the walk's figure would draw a five-minute corridor with fifteen minutes of reach at its edge.
The cell below shows the two figures side by side.

In [ ]:
fixture = json.loads((Path.cwd() / "fixtures" / "band-soho-001.json").read_text())
budget = report["referenceBandBudgetMetres"]
traversal = report["traversalBudgetMetres"]
offered = {j["networkNodeId"]: j for j in fixture["input"]["junctions"]}

print(f"{'junction':<10} {'travelled':>10} {'left of the band':>18} {'left of the traversal':>23}")
for row in sorted(fixture["output"]["inBand"], key=lambda r: -r["remainingM"])[:3]:
    j = offered[row["networkNodeId"]]
    print(f"{row['networkNodeId']:<10} {j['travelMetres']:>10.1f} {row['remainingM']:>18.1f} "
          f"{j['remainingM']:>23.1f}")
for row in sorted(fixture["output"]["inBand"], key=lambda r: r["remainingM"])[:3]:
    j = offered[row["networkNodeId"]]
    print(f"{row['networkNodeId']:<10} {j['travelMetres']:>10.1f} {row['remainingM']:>18.1f} "
          f"{j['remainingM']:>23.1f}")
print()
print(f"The band is {budget:.0f} m and the traversal was {traversal:.0f} m, so every figure in the")
print(f"third column is {traversal - budget:.0f} m larger than the second. Using it would put")
print("fifteen minutes of reach at the edge of a five-minute corridor.")

## What this does not show

- That 75 m a minute is the right walking speed. It is an assumption, estimated from photographs of 1854 London. It
  travels on the junctions so that nothing later can contradict it.
- That the band's edge is where the five-minute line really is. The edge falls between junctions. **Reachable
  corridor** exists to deal with that.